In [0]:
catalog = dbutils.widgets.text("catalog", "hop")
catalog = dbutils.widgets.get("catalog")  

## Order-Event ingestion by Auto loader

In [0]:

from pyspark.sql import functions as F

# 1. CONFIGURATION

source_path = f"/Volumes/{catalog}/bronze/data/raw/order_events/"

checkpoint_path = f"/Volumes/{catalog}/bronze/data/raw/_checkpoints/order_events/"

schema_path = f"/Volumes/{catalog}/bronze/data/raw/_checkpoints/schemas/order_events"

target_table = f"{catalog}.bronze.order_events"

# 2. READ NESTED JSON USING AUTO LOADER

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

# 3. ADD METADATA COLUMNS

df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
# 4. WRITE DATA INTO BRONZE DELTA TABLE
query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True) 
    .toTable(target_table)
)
query.awaitTermination()

# customer inestion by Auto loader

In [0]:
from pyspark.sql import functions as F

# 1. CONFIGURATION

source_path = f"/Volumes/{catalog}/bronze/data/raw/customers/"

checkpoint_path = f"/Volumes/{catalog}/bronze/data/raw/_checkpoints/customers/"

schema_path = f"/Volumes/{catalog}/bronze/data/raw/_checkpoints/schemas/customers"

target_table = f"{catalog}.bronze.customers"

target_table = f"{catalog}.bronze.customers"

# 2. READ NESTED JSON USING AUTO LOADER

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    .option("cloudFiles.inferColumnTypes", "false")
    .option("rescuedDataColumn", "_rescued_data")
    .load(source_path)
)

# 3. ADD METADATA COLUMNS

df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

In [0]:
# 4. WRITE DATA INTO BRONZE DELTA TABLE
query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .trigger(availableNow=True) 
    .toTable(target_table)
)
query.awaitTermination()

# rider ingestion by reader and writer api

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.format("CSV").option("mergeSchema", "true").load(f"/Volumes/{catalog}/bronze/data/raw/riders.csv")
# Add metadata columns
df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)
df.write.mode("overWrite").saveAsTable(f"{catalog}.bronze.riders")

# zones ingestion by reader and writer api

In [0]:
df = spark.read.format("CSV").option("mergeSchema", "true").load(f"/Volumes/{catalog}/bronze/data/raw/zones.csv")
# Add metadata columns
df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)
df.write.mode("overWrite").saveAsTable(f"{catalog}.bronze.zones")

# Support ticket ingestion by reader and writer api

In [0]:
df = spark.read.format("CSV").option("mergeSchema", "true").load(f"/Volumes/{catalog}/bronze/data/raw/support_tickets.csv")
# Add metadata columns
df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)
df.write.mode("overWrite").saveAsTable(f"{catalog}.bronze.support_tickets")

# compensation_probe_tickets ingestion reader api and writer api

In [0]:
df = spark.read.format("CSV").option("mergeSchema", "true").load(f"/Volumes/{catalog}/bronze/data/raw/compensation_probe_tickets.csv")
# Add metadata columns
df = (
    df
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)
df.write.mode("overWrite").saveAsTable(f"{catalog}.bronze.compensation_probe_tickets")

# resturants cdc ingestion with reader and writer api

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:catalog).bronze.restaurants_cdc;

In [0]:
%sql
COPY INTO identifier(:catalog).bronze.restaurants_cdc
FROM (
    SELECT
        *,
        current_timestamp() AS _ingested_at,
        _metadata.file_path AS _source_file
    FROM '/Volumes/' :catalog '/bronze/data/raw/restaurants_cdc/'
)
FILEFORMAT = JSON
COPY_OPTIONS (
    'mergeSchema' = 'true'
)

In [0]:
print("Total rows of All ingested tables")
print("==================================")
print("bronze.order_events: ", spark.table(f"{catalog}.bronze.order_events").count())
print("bronze.customers: ", spark.table(f"{catalog}.bronze.customers").count())
print("bronze.riders: ", spark.table(f"{catalog}.bronze.riders").count())
print("bronze.zones: ", spark.table(f"{catalog}.bronze.zones").count())
print("bronze.support_tickets: ", spark.table(f"{catalog}.bronze.support_tickets").count())
print("bronze.compensation_probe_tickets: ", spark.table(f"{catalog}.bronze.compensation_probe_tickets").count())
print("bronze.restaurants_cdc: ", spark.table(f"{catalog}.bronze.restaurants_cdc").count())